<table border=0 width="100%"><tr><td><p align="left"><font size=5><B>数据分析与可视化</B></font></p></td><td><font size=3><B>OpenCode Skills 开发(郑海超)</B></font></td></tr></table>

# 教学内容
* 从 Prompt 到 Skills：为什么需要封装 AI 工作流
* OpenCode Skill 的结构与创建
* 示例：股评情感分析 Skill
* 为自己的项目创建 Skill

# 回顾：W8 的 Prompt Template

W8 你用 LangChain 的 Prompt Template 约束 LLM 输出：

```python
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["comment"],
    template="判断以下股评的情感倾向（正面/中性/负面），只输出标签：\n{comment}"
)
```

**问题**：
- 每次换任务都要重写 prompt
- AI 不知道你的领域词典、数据格式、分析步骤
- 换个场景（从情感分析换成 EDA 报告），又要从头来

**能不能把 AI 工作流封装成「技能」？** → OpenCode Skills

# 什么是 OpenCode Skill？

Skill = 给 AI 助手的 **SOP（标准操作流程）**

- 不是代码，是一个 markdown 文件（`SKILL.md`）
- 告诉 AI「遇到这类任务该怎么做」
- AI 根据 `description` 自动判断何时加载
- 可以附带参考资料、脚本、模板

## 类比

| 概念 | 代码世界 | AI 世界 |
|---|---|---|
| 函数 | `def analyze(comment):` | SKILL.md 的分析步骤 |
| 参数 | `comment` | 用户输入 |
| 文档 | docstring | description（AI 读它决定何时调用）|
| 配置文件 | `config.yaml` | references/ 目录 |

## 位置

```
你的项目/
└── .opencode/
    └── skills/
        └── 你的skill名/
            ├── SKILL.md          ← 主文件
            └── references/       ← 参考资料（可选）
                └── guide.md
```

# SKILL.md 结构

```markdown
---
name: Skill 名称（简短）
description: 什么时候用这个 Skill（AI 读这段判断是否调用）
---

## 场景
[什么任务、什么数据]

## 步骤
1. 第一步
2. 第二步
3. ...

## 输出格式
[固定格式模板]

## 注意事项
[领域特殊处理]
```

## 两个关键字段

### name
- 简短，用英文或拼音
- 例：`stock-comment-analysis`、`eda-report`、`code-review`

### description ⚠️ 最重要
- AI 根据这段话判断**何时调用**这个 Skill
- 写不好 → AI 不知道何时用 → Skill 白写
- 要写清：**什么任务** + **什么数据**
- 例：`Analyze sentiment of Chinese stock forum comments. Use when the user asks to analyze stock comments or classify financial text sentiment.`

# 示例：股评情感分析 Skill

完整案例在 `cases/.opencode/skills/stock-comment-analysis/`

## SKILL.md 内容

```markdown
---
name: Stock Comment Sentiment Analysis
description: Analyze sentiment of Chinese stock forum comments (股吧评论).
  Use this skill when the user asks to analyze stock comments, classify
  sentiment of financial text, or generate a sentiment summary.
---

## 场景
用户有一批股吧评论，需要判断情感倾向并生成摘要。

## 分析步骤
1. 读取数据，确认评论条数
2. 逐条判断：正面(1) / 中性(0) / 负面(-1)
3. 识别难点：反讽、网络用语、混合情感
4. 生成摘要：情感分布 + 高频话题 + 典型评论

## 输出格式
## 股评情感分析报告
### 概览（总数 + 各情感占比）
### 高频话题（Top 3）
### 典型评论（每类 2-3 条）

## 注意事项
- 金融评论常有反讽（"利好茅台"→负面）
- 短评论(<5字)情感不明显标中性
- 参考情感判断标准：references/sentiment-guide.md
```

## 参考资料

`references/sentiment-guide.md` 包含：
- 情感分类标准（正面/中性/负面的典型特征）
- 常见金融网络用语表（yyds、割韭菜、抄底...）
- 反讽识别规则与示例
- 混合情感处理策略

# 动手：创建 Skill 文件

用 Python 创建 Skill 目录和文件（也可以用文件管理器手动创建）：

In [ ]:
from pathlib import Path

# 在当前项目创建 Skill 目录
skill_dir = Path('.opencode/skills/my-first-skill')
skill_dir.mkdir(parents=True, exist_ok=True)

# 写 SKILL.md
skill_content = '''---
name: My First Skill
description: Describe what this skill does and when to use it.
---

## 场景
[你的 AI 助手反复帮你做什么？]

## 步骤
1. ...
2. ...

## 输出格式
[你希望 AI 按什么格式输出？]
'''

(skill_dir / 'SKILL.md').write_text(skill_content, encoding='utf-8')
print(f'Skill 已创建：{skill_dir / "SKILL.md"}')
print('打开文件编辑内容，然后在 Trae/OpenCode 中测试。')

# Skill 如何改变 AI 行为

## 对比实验

用同一批股评评论（`sse_comments.txt` 前 20 条），让 AI 做情感分析：

### 不带 Skill

直接对 AI 说：「帮我分析这些股评的情感」

结果：
- 输出格式随机（可能是表格、可能是列表、可能没有摘要）
- 可能忽略反讽（"利好茅台"标为正面）
- 不知道你有情感词典
- 每次结果可能不同

### 带 Skill

在 `.opencode/skills/stock-comment-analysis/SKILL.md` 放好后，说同样的：

「帮我分析这些股评的情感」

结果：
- 固定输出格式（概览 + 高频话题 + 典型评论）
- 知道识别反讽和网络用语
- 参考情感判断标准
- 多次调用结果一致

## 课堂演示

教师在 Trae/OpenCode 中：
1. 先不带 Skill 让 AI 分析股评 → 看输出
2. 把 `cases/.opencode/skills/stock-comment-analysis/` 复制到项目
3. 再让 AI 分析同一批股评 → 对比输出

# 设计原则：怎么写好 Skill

## 1. description 是灵魂

❌ 不好：`description: 分析文本`
✅ 好：`description: Analyze sentiment of Chinese stock forum comments. Use when the user asks to analyze stock comments or classify financial text sentiment.`

原则：写清**什么任务** + **什么数据** + **什么时候用**

## 2. 步骤要具体

❌ 不好：`分析评论的情感`
✅ 好：`1. 读取数据 2. 逐条判断正面/中性/负面 3. 识别反讽 4. 生成摘要`

## 3. 输出格式要固定

给一个模板，AI 照着填。不固定格式 = 每次不一样。

## 4. 注意事项写领域知识

把你的领域知识写进去——这正是 Skill 的价值：让通用 AI 有领域专家的判断力。

## 5. 参考资料分离

大段规则放 `references/` 目录，SKILL.md 保持简洁。AI 会自动读取参考资料。

## 迭代改进

和写代码一样：先写最简版 → 用一次 → 发现问题 → 补充 → 再用

- 第一次输出格式不对？→ 补充输出格式模板
- AI 忽略反讽？→ 在注意事项里加反讽规则
- AI 不知道词典？→ 放参考资料文件

# 为自己的项目创建 Skill

## 第一步：想清楚你的 AI 助手反复帮你做什么？

常见场景：

| 你的项目类型 | Skill 想法 |
|---|---|
| 股评分析 | 股评情感分析 Skill |
| 基金分析 | 基金业绩评估 Skill |
| 经济数据分析 | EDA 报告生成 Skill |
| 文本分析 | 文本预处理 Skill（分词+去停用词+向量化）|
| 数据可视化 | 图表推荐 Skill（根据数据类型推荐图表）|
| 交易策略 | 策略评估 Skill（回测指标计算）|

## 第二步：创建目录和文件

在项目根目录创建：
```
.opencode/skills/你的skill名/SKILL.md
```

## 第三步：写好 frontmatter + 指令

## 第四步：用一次，看输出对不对

## 第五步：迭代改进

# 课堂练习

用 `sse_comments.txt` 的前 20 条评论：

1. 不带 Skill，让 AI 分析 → 记录输出格式和质量
2. 把 `cases/.opencode/skills/stock-comment-analysis/` 复制到项目
3. 带 Skill，让 AI 分析同一批评论 → 对比
4. 思考：你的项目需要什么 Skill？

# 课后作业

1. 为自己的项目创建 ≥1 个 OpenCode Skill
2. 完成 W9 技能小练（互评校准）
3. 完成组内互评
4. 回看 `llm.ipynb` 中 Attention/Transformer 部分